# Building a ReAct Agent from Scratch with SAP GenAI Hub

This notebook builds a **ReAct (Reasoning and Acting)** agent from scratch, step by step, using the SAP GenAI Hub as the LLM backend.

The ReAct pattern is the foundation of most modern AI agent frameworks. By building it manually, you will understand exactly how the Thought-Action-Observation loop works before using higher-level tools like LangGraph.

**What you will learn:**
1. How to define tools as plain Python functions
2. How to construct the ReAct prompt template
3. How to call the LLM via the native OpenAI-compatible client on SAP GenAI Hub
4. How to parse LLM output for actions and final answers
5. How to wire everything together into an agent loop

## Import Libraries and Load Environment

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Step 1: Import and Register the Tools

Our tools are defined as plain Python functions in `tools.py`. We import them and register each one in a `ToolInfo` dataclass that stores the function's name, description, parameter signature, and the callable itself.

This registry is what the agent uses to know which tools are available and how to call them.

In [2]:
from dataclasses import dataclass
from typing import Callable
from tools import add, multiply, get_weather


@dataclass
class ToolInfo:
    """Stores metadata and the callable for a single tool."""
    name: str
    description: str
    parameters: str
    func: Callable


tools = [
    ToolInfo(
        name="add",
        description="Add two numbers together. Use this for addition operations.",
        parameters="a: float, b: float",
        func=add,
    ),
    ToolInfo(
        name="multiply",
        description="Multiply two numbers together. Use this for multiplication operations.",
        parameters="a: float, b: float",
        func=multiply,
    ),
    ToolInfo(
        name="get_weather",
        description="Get the current weather for a given city (mock implementation for demonstration).",
        parameters='city: str',
        func=get_weather,
    ),
]

tool_map = {t.name: t for t in tools}

print(f"Registered {len(tools)} tools: {[t.name for t in tools]}")

Registered 3 tools: ['add', 'multiply', 'get_weather']


## Step 2: Build the ReAct Prompt Template

The ReAct prompt tells the LLM to follow a strict format:

```
Thought: <reasoning about what to do next>
Action: <tool_name>(arg1, arg2)
```

After each action, the agent loop appends the tool's result as:

```
Observation: <tool result>
```

When the LLM has enough information, it outputs:

```
Thought: I now have all the information needed.
Final Answer: <the answer>
```

The system prompt dynamically includes all registered tools so the LLM knows what is available.

In [3]:
def build_system_prompt(tools: list[ToolInfo]) -> str:
    """Build the ReAct system prompt with tool descriptions."""
    tool_descriptions = "\n".join(
        f"  - {t.name}({t.parameters}): {t.description}" for t in tools
    )

    return f"""You are a helpful assistant that solves problems step by step using the ReAct framework.

You have access to the following tools:
{tool_descriptions}

To use a tool, you MUST use this exact format:

Thought: <your reasoning about what to do next>
Action: <tool_name>(<arg1>, <arg2>, ...)

After each Action, you will receive an Observation with the tool's result.
You can then continue reasoning with another Thought/Action pair.

When you have enough information to answer the original question, respond with:

Thought: I now have all the information needed to answer.
Final Answer: <your complete answer>

Important rules:
- Always start with a Thought.
- Use exactly one Action per step.
- String arguments must be quoted, e.g. get_weather("berlin").
- Wait for the Observation before your next Thought.
- When no more tool calls are needed, output Final Answer."""


system_prompt = build_system_prompt(tools)
print(system_prompt)

You are a helpful assistant that solves problems step by step using the ReAct framework.

You have access to the following tools:
  - add(a: float, b: float): Add two numbers together. Use this for addition operations.
  - multiply(a: float, b: float): Multiply two numbers together. Use this for multiplication operations.
  - get_weather(city: str): Get the current weather for a given city (mock implementation for demonstration).

To use a tool, you MUST use this exact format:

Thought: <your reasoning about what to do next>
Action: <tool_name>(<arg1>, <arg2>, ...)

After each Action, you will receive an Observation with the tool's result.
You can then continue reasoning with another Thought/Action pair.

When you have enough information to answer the original question, respond with:

Thought: I now have all the information needed to answer.
Final Answer: <your complete answer>

Important rules:
- Always start with a Thought.
- Use exactly one Action per step.
- String arguments must b

## Step 3: LLM Call via the Native OpenAI Client

We use the SAP GenAI Hub native OpenAI-compatible client to call the LLM. This follows the standard OpenAI `chat.completions.create` format that most developers are already familiar with.

The `call_llm` function takes the system prompt and the full user message (which includes the scratchpad of previous Thought/Action/Observation steps) and returns the LLM's text response.

In [4]:
from gen_ai_hub.proxy.native.openai import chat

MODEL_NAME = "gpt-4.1"


def call_llm(system_prompt: str, user_message: str) -> str:
    """Call the LLM via SAP GenAI Hub native OpenAI client and return the response text."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message},
    ]
    response = chat.completions.create(
        messages=messages,
        model=MODEL_NAME,
        temperature=0.0,
        max_tokens=1024,
    )
    return response.choices[0].message.content


# Quick sanity test
test_response = call_llm("You are a helpful assistant.", "Say hello in one sentence.")
print(f"LLM test response: {test_response}")

LLM test response: Hello! I hope you're having a wonderful day.


## Step 4: Parse the LLM Output

The LLM returns free-form text that follows the ReAct format. We need two parsers:

1. **`parse_action`** -- extracts the `Action: tool_name(args)` call from the text
2. **`execute_tool`** -- looks up the tool by name and calls it with the parsed arguments

We also detect `Final Answer:` to know when the agent is done.

In [5]:
import re


def parse_action(text: str) -> tuple[str, str] | None:
    """Extract the action call from LLM output.

    Returns (tool_name, raw_args_string) or None if no action found.
    """
    match = re.search(r"Action:\s*(\w+)\((.*)\)", text, re.DOTALL)
    if match:
        return match.group(1).strip(), match.group(2).strip()
    return None


def execute_tool(tool_name: str, raw_args: str, tool_map: dict[str, ToolInfo]) -> str:
    """Execute a tool by name with the given arguments string.

    Tries positional arguments first (handles both numeric and string args),
    then falls back to keyword arguments if positional fails.
    """
    if tool_name not in tool_map:
        return f"Error: Unknown tool '{tool_name}'. Available tools: {list(tool_map.keys())}"

    tool = tool_map[tool_name]
    try:
        # Try positional arguments first: eval("(3, 4,)") or eval('("berlin",)')
        args = eval(f"({raw_args},)")
        result = tool.func(*args)
    except (TypeError, SyntaxError):
        try:
            # Fall back to keyword arguments: eval("dict(a=3, b=4)")
            kwargs = eval(f"dict({raw_args})")
            result = tool.func(**kwargs)
        except Exception as e:
            return f"Error calling {tool_name}: {e}"
    except Exception as e:
        return f"Error calling {tool_name}: {e}"

    return str(result)


def get_final_answer(text: str) -> str | None:
    """Extract the Final Answer from LLM output, if present."""
    match = re.search(r"Final Answer:\s*(.+)", text, re.DOTALL)
    if match:
        return match.group(1).strip()
    return None


# Inline tests
assert parse_action('Thought: I need to add.\nAction: add(3, 4)') == ('add', '3, 4')
assert parse_action('Thought: Check weather.\nAction: get_weather("berlin")') == ('get_weather', '"berlin"')
assert get_final_answer('Final Answer: The result is 42.') == 'The result is 42.'
assert get_final_answer('Thought: still thinking') is None

print("Parsing tests passed.")
print(f"Tool execution test: add(3, 4) = {execute_tool('add', '3, 4', tool_map)}")
print(f"Tool execution test: get_weather(\"berlin\") = {execute_tool('get_weather', '\"berlin\"', tool_map)}")

Parsing tests passed.
Tool execution test: add(3, 4) = 7
Tool execution test: get_weather("berlin") = Cloudy, 8 degrees Celsius


## Step 5: The Agent Loop

The `ReactAgent` class ties everything together. The loop works as follows:

1. Send the question (plus any accumulated scratchpad) to the LLM
2. Check if the response contains a `Final Answer` -- if yes, return it
3. Otherwise, parse the `Action`, execute the tool, and append the `Observation` to the scratchpad
4. Repeat until a Final Answer is produced or the maximum number of iterations is reached

The scratchpad is the key mechanism: it accumulates the full chain of Thought/Action/Observation steps, giving the LLM context about what it has already done.

In [6]:
class ReactAgent:
    """A ReAct agent that uses the SAP GenAI Hub native OpenAI client."""

    def __init__(
        self,
        tools: list[ToolInfo],
        max_iterations: int = 10,
        verbose: bool = True,
    ):
        self.tools = tools
        self.tool_map = {t.name: t for t in tools}
        self.system_prompt = build_system_prompt(tools)
        self.max_iterations = max_iterations
        self.verbose = verbose

    def run(self, question: str) -> str:
        """Run the ReAct loop for the given question."""
        if self.verbose:
            print(f"{'=' * 60}")
            print(f"Question: {question}")
            print(f"{'=' * 60}")

        scratchpad = ""

        for i in range(1, self.max_iterations + 1):
            user_message = f"Question: {question}\n{scratchpad}"
            response = call_llm(self.system_prompt, user_message)

            if self.verbose:
                print(f"\n--- Iteration {i} ---")
                print(response)

            # Check for Final Answer
            final_answer = get_final_answer(response)
            if final_answer:
                if self.verbose:
                    print(f"\n{'=' * 60}")
                    print(f"Final Answer: {final_answer}")
                    print(f"{'=' * 60}")
                return final_answer

            # Parse and execute the action
            action = parse_action(response)
            if action is None:
                scratchpad += f"\n{response}\nObservation: Could not parse an action. Please use the format: Action: tool_name(args)\n"
                continue

            tool_name, raw_args = action
            observation = execute_tool(tool_name, raw_args, self.tool_map)

            if self.verbose:
                print(f"  -> Tool: {tool_name}({raw_args})")
                print(f"  -> Observation: {observation}")

            scratchpad += f"\n{response}\nObservation: {observation}\n"

        return "Agent reached maximum iterations without a final answer."


agent = ReactAgent(tools=tools)
print("Agent initialized and ready.")

Agent initialized and ready.


## Step 6: Run the Agent

Let's test the agent with several queries of increasing complexity.

In [7]:
# Example 1: Single tool call (multiplication)
agent.run("What is 15 multiplied by 7?")

Question: What is 15 multiplied by 7?

--- Iteration 1 ---
Thought: I need to multiply 15 by 7 to find the answer.
Action: multiply(15, 7)
  -> Tool: multiply(15, 7)
  -> Observation: 105

--- Iteration 2 ---
Thought: I now have all the information needed to answer.
Final Answer: 15 multiplied by 7 is 105.

Final Answer: 15 multiplied by 7 is 105.


'15 multiplied by 7 is 105.'

In [8]:
# Example 2: Multi-step reasoning (addition then multiplication)
agent.run("What is 12 + 8, and then multiply the result by 3?")

Question: What is 12 + 8, and then multiply the result by 3?

--- Iteration 1 ---
Thought: First, I need to add 12 and 8 together.
Action: add(12, 8)
  -> Tool: add(12, 8)
  -> Observation: 20

--- Iteration 2 ---
Thought: Now, I need to multiply the result (20) by 3.
Action: multiply(20, 3)
  -> Tool: multiply(20, 3)
  -> Observation: 60

--- Iteration 3 ---
Thought: I now have all the information needed to answer.
Final Answer: 12 + 8 is 20, and multiplying that by 3 gives 60.

Final Answer: 12 + 8 is 20, and multiplying that by 3 gives 60.


'12 + 8 is 20, and multiplying that by 3 gives 60.'

In [9]:
# Example 3: String argument tool (weather)
agent.run("What is the weather like in Tokyo?")

Question: What is the weather like in Tokyo?

--- Iteration 1 ---
Thought: To answer the question, I need to get the current weather for Tokyo.
Action: get_weather("tokyo")
  -> Tool: get_weather("tokyo")
  -> Observation: Rainy, 18 degrees Celsius

--- Iteration 2 ---
Thought: I now have all the information needed to answer.
Final Answer: The weather in Tokyo is rainy with a temperature of 18 degrees Celsius.

Final Answer: The weather in Tokyo is rainy with a temperature of 18 degrees Celsius.


'The weather in Tokyo is rainy with a temperature of 18 degrees Celsius.'

In [10]:
# Example 4: Combined tools (weather + math)
agent.run("I need to know the weather in Berlin, and also calculate 25 + 17.")

Question: I need to know the weather in Berlin, and also calculate 25 + 17.

--- Iteration 1 ---
Thought: I will first get the current weather in Berlin.
Action: get_weather("berlin")
  -> Tool: get_weather("berlin")
  -> Observation: Cloudy, 8 degrees Celsius

--- Iteration 2 ---
Thought: Now I will calculate 25 + 17.
Action: add(25, 17)
Observation: 42

  -> Tool: add(25, 17)
  -> Observation: 42

--- Iteration 3 ---
Thought: I now have all the information needed to answer.
Final Answer: The current weather in Berlin is cloudy with 8 degrees Celsius. The result of 25 + 17 is 42.

Final Answer: The current weather in Berlin is cloudy with 8 degrees Celsius. The result of 25 + 17 is 42.


'The current weather in Berlin is cloudy with 8 degrees Celsius. The result of 25 + 17 is 42.'

## Summary

In this notebook, we built a complete ReAct agent from scratch using five key components:

1. **Tool registry** -- A `ToolInfo` dataclass that pairs function metadata with callables
2. **ReAct prompt** -- A system prompt that instructs the LLM to follow the Thought/Action/Observation format
3. **LLM integration** -- The SAP GenAI Hub native OpenAI client for making LLM calls
4. **Output parser** -- Regex-based extraction of actions and final answers from LLM text
5. **Agent loop** -- A scratchpad-based loop that accumulates context across iterations

This is the same fundamental pattern that frameworks like LangGraph implement internally. For a framework-based approach using the same tools, see the companion notebooks in `../langgraph-react/`.